# Photon catalysis
Reproducing and improving the maximum success probabilities found in Table III of Aralov et al., PRX Quantum 7, 020323 (2026).

In [1]:
import numpy as np

from qoptcraft import Fock, PureState
from qoptcraft.optimization import PhotonCatalysisStatePrep, PiecewiseInfidelityCost, BFGS

import sys, os
sys.path.insert(0, os.path.abspath('..'))
from results_table import examples_table

In [2]:
optimizer = BFGS(max_iter=2000, line_search="wolfe")

In [3]:
r3, r6 = np.sqrt(3), np.sqrt(6)

STATES = {
    "Psi1":  dict(target=Fock(2,0,0) + Fock(0,2,0) + Fock(0,0,2), additions=3, herald=1, paper=0.29),
    "Psi2":  dict(target=Fock(3,0,0) + Fock(0,3,0) + Fock(0,0,3), additions=4, herald=1, paper=0.27),
    "Psi3":  dict(target=Fock(4,0,0) + Fock(0,4,0) + Fock(0,0,4), additions=6, herald=2, paper=0.19),
    "Psi4":  dict(target=Fock(2,0,0,0) + Fock(0,2,0,0) + Fock(0,0,2,0) + Fock(0,0,0,2),
                  additions=4, herald=2, paper=0.21),
    "Psi5":  dict(target=Fock(0,1,2) + Fock(1,2,0) + Fock(2,0,1)
                       + Fock(0,2,1) + Fock(1,0,2) + Fock(2,1,0), additions=4, herald=1, paper=0.29),
    "Psi6":  dict(target=Fock(1,1,0) + Fock(1,0,1) + Fock(0,1,1), additions=3, herald=1, paper=0.32),
    "Psi7":  dict(target=Fock(2,2,0) + Fock(2,0,2) + Fock(0,2,2), additions=5, herald=1, paper=0.30),
    "Psi8":  dict(target=Fock(2,0,0,0) + Fock(0,1,1,0) + Fock(0,0,0,2),
                  additions=4, herald=2, paper=0.22),
    "Psi9":  dict(target=Fock(3,0,0,0) + Fock(0,2,1,0) + Fock(0,1,2,0) + Fock(0,0,0,3),
                  additions=5, herald=2, paper=0.18),
    "Psi10": dict(target=Fock(0,4,0) + Fock(1,2,1) + Fock(2,0,2), additions=6, herald=2, paper=0.14),
    "R2":    dict(target=PureState([(3,0,0), (1,2,0), (1,1,1), (1,0,2)], [1, r3, r6, r3]),
                  additions=4, herald=1, paper=0.82),
    "R4":    dict(target=Fock(3,0,0) + Fock(0,3,0) + Fock(0,0,3) + Fock(1,1,1),
                  additions=4, herald=1, paper=0.30),
    "R5":    dict(target=Fock(2,1,0) + Fock(0,2,1), additions=4, herald=1, paper=0.31),
    "K3":    dict(target=Fock(3,0,0,0) + Fock(2,1,0,0) + Fock(2,0,1,0) + Fock(2,0,0,1)
                       - Fock(1,1,1,0) - Fock(1,1,0,1) - Fock(1,0,1,1) - Fock(0,1,1,1),
                  additions=5, herald=2, paper=0.18),
}

In [ ]:
def build_problem(state):
    """Vacuum on `target.modes + 1` modes -> photon additions on the ancilla -> PNR herald."""
    modes = state["target"].modes
    return PhotonCatalysisStatePrep(
        Fock(*[0] * (modes + 1)), state["target"],
        add_mode=modes, n_additions=state["additions"], herald=state["herald"],
    )


problems = {name: build_problem(state) for name, state in STATES.items()}

In [ ]:
N_RUNS = 4000
all_results = {}
for name, problem in problems.items():
    cost_fun = PiecewiseInfidelityCost(problem)
    all_results[name] = [
        optimizer.minimize(cost_fun, cost_fun.manifold.random_point(), verbose=False)
        for _ in range(N_RUNS)
    ]
    print(f"Optimization of {name} finished.")

In [ ]:
table = examples_table(all_results, problems, time_format=".4f", index_name="State")
table["Paper (%)"] = [f"{100 * state['paper']:.0f}%" for state in STATES.values()]
table

,Runs,Converged (%),Best prob (%),Mean time (s),Mean iters,Paper (%)
State,,,,,,
Psi1,4000,100.0%,29.601%,0.0825,101,29%
Psi2,4000,99.7%,27.558%,0.4917,309,27%
Psi3,4000,96.9%,22.547%,2.8703,655,19%
Psi4,4000,100.0%,21.603%,0.2475,146,21%
Psi5,4000,99.5%,30.033%,0.4149,329,29%
Psi6,4000,100.0%,34.647%,0.1629,248,32%
Psi7,4000,85.2%,30.102%,1.5227,683,30%
Psi8,4000,100.0%,22.641%,0.8164,495,22%
Psi9,4000,45.2%,18.977%,0.5394,150,18%
